In [1]:
import pandas as pd

DATA_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

# Load raw data
orders = pd.read_csv(DATA_PATH + "orders.csv")
order_products_prior = pd.read_csv(DATA_PATH + "order_products__prior.csv")
products = pd.read_csv(DATA_PATH + "products.csv")
departments = pd.read_csv(DATA_PATH + "departments.csv")

# Load customer segmentation result
segments = pd.read_csv(
    PROCESSED_PATH + "customer_segments_without_produce_dairy.csv",
    index_col=0
)

# Keep only useful order columns
user_orders = orders[
    [
        "order_id",
        "user_id",
        "order_hour_of_day",
        "order_dow"
    ]
]

# Merge order products with user/order time info
analysis_table = order_products_prior.merge(
    user_orders,
    on="order_id",
    how="left"
)

# Merge product info
analysis_table = analysis_table.merge(
    products[
        [
            "product_id",
            "product_name",
            "department_id"
        ]
    ],
    on="product_id",
    how="left"
)

# Merge department name
analysis_table = analysis_table.merge(
    departments,
    on="department_id",
    how="left"
)

# Merge cluster label
analysis_table = analysis_table.merge(
    segments[["cluster"]],
    left_on="user_id",
    right_index=True,
    how="left"
)

# Keep only useful columns
analysis_table = analysis_table[
    [
        "user_id",
        "cluster",
        "order_id",
        "product_name",
        "department",
        "order_hour_of_day",
        "order_dow",
        "reordered"
    ]
]

analysis_table.head()

,user_id,cluster,order_id,product_name,department,order_hour_of_day,order_dow,reordered
0,202279,4,2,Organic Egg Whites,dairy eggs,9,5,1
1,202279,4,2,Michigan Organic Kale,produce,9,5,1
2,202279,4,2,Garlic Powder,pantry,9,5,0
3,202279,4,2,Coconut Butter,pantry,9,5,1
4,202279,4,2,Natural Sweetener,pantry,9,5,0


In [2]:
analysis_table.to_csv(
    PROCESSED_PATH + "cluster_analysis_table.csv",
    index=False
)

print("Saved cluster_analysis_table.csv")
print(analysis_table.shape)

Saved cluster_analysis_table.csv
(32434489, 8)
